In [15]:
import pandas as pd

#Load the dataset
df = pd.read_csv('cirrhosis.csv')

In [16]:
import numpy as np
import tensorflow as tf

np.random.seed(42)
tf.random.set_seed(42)

In [17]:
#preview before cleaning data
print("Data before cleaning:"), df.shape
print(df.isnull().sum())

Data before cleaning:
ID                 0
N_Days             0
Status             0
Drug             106
Age                0
Sex                0
Ascites          106
Hepatomegaly     106
Spiders          106
Edema              0
Bilirubin          0
Cholesterol      134
Albumin            0
Copper           108
Alk_Phos         106
SGOT             106
Tryglicerides    136
Platelets         11
Prothrombin        2
Stage              6
dtype: int64


In [18]:
#Remove all rows with missing data
df = df.dropna()

In [19]:
#confirm result
print("\nShape after removing missing data:", df.shape)
print(df.head())


Shape after removing missing data: (276, 20)
   ID  N_Days Status             Drug    Age Sex Ascites Hepatomegaly Spiders  \
0   1     400      D  D-penicillamine  21464   F       Y            Y       Y   
1   2    4500      C  D-penicillamine  20617   F       N            Y       Y   
2   3    1012      D  D-penicillamine  25594   M       N            N       N   
3   4    1925      D  D-penicillamine  19994   F       N            Y       Y   
4   5    1504     CL          Placebo  13918   F       N            Y       Y   

  Edema  Bilirubin  Cholesterol  Albumin  Copper  Alk_Phos    SGOT  \
0     Y       14.5        261.0     2.60   156.0    1718.0  137.95   
1     N        1.1        302.0     4.14    54.0    7394.8  113.52   
2     S        1.4        176.0     3.48   210.0     516.0   96.10   
3     S        1.8        244.0     2.54    64.0    6121.8   60.63   
4     N        3.4        279.0     3.53   143.0     671.0  113.15   

   Tryglicerides  Platelets  Prothrombin  Stag

In [20]:
#drop columns, ID, N_Days, Status, Drug
X = df.drop(columns=['ID', 'N_Days', 'Status', 'Drug'])
display(X.head())

,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage
0,21464,F,Y,Y,Y,Y,14.5,261.0,2.60,156.0,1718.0,137.95,172.0,190.0,12.2,4.0
1,20617,F,N,Y,Y,N,1.1,302.0,4.14,54.0,7394.8,113.52,88.0,221.0,10.6,3.0
2,25594,M,N,N,N,S,1.4,176.0,3.48,210.0,516.0,96.10,55.0,151.0,12.0,4.0
3,19994,F,N,Y,Y,S,1.8,244.0,2.54,64.0,6121.8,60.63,92.0,183.0,10.3,4.0
4,13918,F,N,Y,Y,N,3.4,279.0,3.53,143.0,671.0,113.15,72.0,136.0,10.9,3.0


In [21]:
#Convert text to numerical data - One-hot encoding

X = pd.get_dummies(X, columns=['Edema'])


In [22]:
#use lambda function to apply a simple rule to each value
y = df['Status'].apply(lambda x: 0 if x == 'D' else 1)
display(y.head())

,Status
0,0
1,1
2,0
3,0
4,1


In [23]:
X = X.replace('F', 1).replace('M', 0) #replace sex value to numerical
X = X.replace('Y', 1).replace('N', 0) #replace Y/N to numerical
X = X.astype(float) #convert to float

/tmp/ipykernel_6502/1980880131.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X = X.replace('F', 1).replace('M', 0) #replace sex value to numerical
/tmp/ipykernel_6502/1980880131.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X = X.replace('Y', 1).replace('N', 0) #replace Y/N to numerical


## **Training and testing the model**

In [24]:
#split 276 rows into 80% training and 20% testing. random_state=42 locks the shuffle so it is the same split each time you run it
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [25]:
from sklearn.preprocessing import StandardScaler

#scale the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [26]:
import tensorflow as tf
from tensorflow import keras

model = keras.Sequential([
    keras.layers.Input(shape=(X_train.shape[1],)), #tells model how many input featues to expect
    keras.layers.Dense(16, activation='relu'), #2 hidden layers with 16 neurons each
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid') #squishes output to value of 0 and 1
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

#epoch: how many times the model passes through the traning set
history = model.fit(X_train, y_train, epochs=10, verbose=1)

Epoch 1/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.3818 - loss: 0.8173
Epoch 2/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.4136 - loss: 0.7705
Epoch 3/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.4909 - loss: 0.7324
Epoch 4/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5818 - loss: 0.7010 
Epoch 5/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6318 - loss: 0.6748
Epoch 6/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6682 - loss: 0.6525
Epoch 7/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6818 - loss: 0.6332
Epoch 8/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.6864 - loss: 0.6157
Epoch 9/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.7091 - loss: 0.5995
Epoch 10/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.7364 - loss: 0.5847


In [27]:
from sklearn.metrics import accuracy_score

final_loss = history.history['loss'][-1] #grabs the value from the last epoch
final_acc = history.history['accuracy'][-1]

y_pred = (model.predict(X_test) > 0.5).astype(int).flatten()
test_acc = accuracy_score(y_test, y_pred)

print(f"Final Training Loss: {final_loss}")
print(f"Final Training Accuracy: {final_acc}")
print(f"Final Testing Accuracy: {test_acc}")

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step
Final Training Loss: 0.5847201347351074
Final Training Accuracy: 0.7363636493682861
Final Testing Accuracy: 0.8035714285714286
